# Tarea 03 - AG: Aplicación práctica — Problema del Viajante de Comercio (TSP)


---

## Descripción del problema

El **Problema del Viajante de Comercio (TSP)** consiste en buscar el recorrido de longitud mínima
que debe seguir un comerciante para visitar un conjunto de ciudades y volver al punto de partida,
sin pasar dos veces por la misma ciudad.

- **Codificación:** Permutación de enteros (1 a N), donde cada permutación representa un recorrido.
- **Función de costo:** Suma de distancias recorridas entre ciudades consecutivas (incluyendo regreso al origen).
- **Objetivo:** Minimizar la distancia total del recorrido.
- **Ciudades:** 15 ciudades con posiciones en un plano 2D.
- **Visualización:** Se grafica la mejor solución encontrada a lo largo de las generaciones.

---
## 1. Importaciones y configuración

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pygad
import math
import random
import copy
import warnings
warnings.filterwarnings("ignore")

%matplotlib inline
plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

print("Importaciones completadas.")

---
## 2. Definición de las ciudades

Se definen **15 ciudades** ubicadas en un plano 2D. Para facilitar la evaluación,
se disponen alrededor de un círculo (configuración conocida donde la solución óptima
es el perímetro del círculo) y se añaden algunas ciudades interiores para hacer el
problema más interesante.

In [ ]:
NUM_CITIES = 15
CITY_RADIUS = 10

np.random.seed(42)
cities = []

for i in range(12):
    angle = 2 * np.pi * i / 12
    x = CITY_RADIUS * np.cos(angle)
    y = CITY_RADIUS * np.sin(angle)
    cities.append((x, y))

cities.append((3.0, 2.0))
cities.append((-4.0, 3.0))
cities.append((1.0, -5.0))

cities = np.array(cities)

city_labels = list(range(1, NUM_CITIES + 1))

print(f"Número de ciudades: {NUM_CITIES}")
print(f"Posiciones de las ciudades:")
for i, (x, y) in enumerate(cities):
    print(f"  Ciudad {i+1}: ({x:.2f}, {y:.2f})")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(cities[:, 0], cities[:, 1], c='red', s=100, zorder=5, edgecolors='black')
for i, (x, y) in enumerate(cities):
    ax.annotate(f'{i+1}', (x, y), textcoords="offset points", xytext=(8, 8),
                fontsize=11, fontweight='bold')
ax.set_xlabel('X', fontsize=12)
ax.set_ylabel('Y', fontsize=12)
ax.set_title('Ubicación de las 15 ciudades', fontsize=14)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 3. Matriz de distancias

Se calcula la distancia Euclidiana entre cada par de ciudades.

In [ ]:
def compute_distance_matrix(cities):
    n = len(cities)
    dist_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            dist_matrix[i][j] = np.linalg.norm(cities[i] - cities[j])
    return dist_matrix

dist_matrix = compute_distance_matrix(cities)

print("Matriz de distancias (redondeada a 2 decimales):")
print(np.round(dist_matrix, 2))

---
## 4. Función de aptitud (fitness)

La función de costo es la **distancia total** del recorrido. PyGAD **maximiza** la aptitud,
por lo que usamos `fitness = 1 / distancia_total` para convertir la minimización en maximización.

In [ ]:
def route_distance(route, dist_matrix):

    total = 0.0
    n = len(route)
    for i in range(n):
        city_from = int(route[i])
        city_to = int(route[(i + 1) % n])
        total += dist_matrix[city_from][city_to]
    return total


def fitness_tsp(ga_instance, solution, solution_idx):

    distance = route_distance(solution, dist_matrix)
    if distance == 0:
        return float('inf')
    return 1.0 / distance


print("Función de aptitud definida.")
print("  Estrategia: fitness = 1 / distancia_total")

---
## 5. Operadores genéticos para permutaciones

El TSP requiere operadores especiales que preserven la propiedad de **permutación**
(cada ciudad aparece exactamente una vez en la ruta). Los operadores estándar de
cruce y mutación pueden generar soluciones inválidas.

- **Cruce:** Order Crossover (OX) — preserva el orden relacional de las ciudades.
- **Mutación:** Inversión de subrutas (swap de dos ciudades) — preserva la permutación.

In [ ]:
def order_crossover(parents, offspring_size, ga_instance):

    offspring = []
    num_offspring = offspring_size[0]
    num_genes = offspring_size[1]

    for k in range(num_offspring):
        parent1 = list(parents[k % parents.shape[0]])
        parent2 = list(parents[(k + 1) % parents.shape[0]])

        cut_points = sorted(random.sample(range(num_genes), 2))
        start, end = cut_points

        child = [-1] * num_genes
        for i in range(start, end + 1):
            child[i] = parent1[i]

        segment_cities = set(child[start:end + 1])
        remaining = [city for city in parent2 if city not in segment_cities]

        idx = (end + 1) % num_genes
        for city in remaining:
            while child[idx] != -1:
                idx = (idx + 1) % num_genes
            child[idx] = city

        offspring.append(child)

    return np.array(offspring)


def swap_mutation(offspring, ga_instance):

    pm = ga_instance.mutation_probability
    if pm is None:
        pm = 0.1

    for chromosome_idx in range(offspring.shape[0]):
        if np.random.random() < pm:
            idx1, idx2 = random.sample(range(offspring.shape[1]), 2)
            (offspring[chromosome_idx, idx1],
             offspring[chromosome_idx, idx2]) = (offspring[chromosome_idx, idx2],
                                                  offspring[chromosome_idx, idx1])
    return offspring


print("Operadores genéticos para permutaciones definidos:")
print("  Cruce: Order Crossover (OX)")
print("  Mutación: Swap (intercambio de dos ciudades)")

---
## 6. Callback para registrar la evolución

En cada generación se registra:
- La distancia de la mejor ruta encontrada
- La distancia promedio de la población

También se almacenan las mejores rutas cada cierto número de generaciones
para su visualización posterior.

In [ ]:
best_distance_history = []
avg_distance_history = []
best_routes_log = {}
SAVE_ROUTE_EVERY = 10

def on_generation_tsp(ga_instance):
    global best_distance_history, avg_distance_history, best_routes_log

    best_solution, best_fitness, _ = ga_instance.best_solution()
    best_dist = route_distance(best_solution, dist_matrix)
    best_distance_history.append(best_dist)

    all_fitness = ga_instance.last_generation_fitness
    all_distances = [route_distance(ga_instance.population[i], dist_matrix)
                     for i in range(len(ga_instance.population))]
    avg_distance_history.append(np.mean(all_distances))

    gen = ga_instance.generations_completed
    if gen % SAVE_ROUTE_EVERY == 0 or gen == 1:
        best_routes_log[gen] = best_solution.copy()

print("Callback de evolución definido.")
print(f"  Se guardará la mejor ruta cada {SAVE_ROUTE_EVERY} generaciones.")

---
## 7. Configuración y ejecución del Algoritmo Genético

Parámetros del AG para el TSP:

| Parámetro | Valor |
|---|---|
| Población | 100 |
| Generaciones | 500 |
| Padres para cruce | 50 |
| Selección | Rueda de ruleta |
| Cruce | Order Crossover (OX), Pc=0.9 |
| Mutación | Swap, Pm=0.1 |
| Elitismo | 2 individuos |

In [ ]:
best_distance_history = []
avg_distance_history = []
best_routes_log = {}

POP_SIZE = 100
NUM_GENERATIONS = 500
NUM_PARENTS_MATING = 50
PC = 0.9
PM = 0.1
KEEP_ELITISM = 2

initial_population = []
for _ in range(POP_SIZE):
    individual = list(range(NUM_CITIES))
    random.shuffle(individual)
    initial_population.append(individual)
initial_population = np.array(initial_population)

ga_tsp = pygad.GA(
    num_generations=NUM_GENERATIONS,
    num_parents_mating=NUM_PARENTS_MATING,
    initial_population=initial_population,
    fitness_func=fitness_tsp,
    parent_selection_type="rws",
    crossover_type=order_crossover,
    crossover_probability=PC,
    mutation_type=swap_mutation,
    mutation_probability=PM,
    keep_elitism=KEEP_ELITISM,
    allow_duplicate_genes=True,
    on_generation=on_generation_tsp,
    suppress_warnings=True,
    random_seed=42,
    gene_type=int,
)

print("Algoritmo Genético configurado.")
print(f"  Población: {POP_SIZE}")
print(f"  Generaciones: {NUM_GENERATIONS}")
print(f"  Pc={PC}, Pm={PM}")
print(f"  Elitismo: {KEEP_ELITISM}")

In [ ]:
ga_tsp.run()

print("Ejecución completada.")

---
## 8. Mejor solución encontrada

Se muestra la mejor ruta encontrada, su distancia total y se compara
con la distancia promedio inicial y final.

In [ ]:
best_solution, best_fitness, best_idx = ga_tsp.best_solution()
best_route = best_solution.astype(int)
best_dist = route_distance(best_route, dist_matrix)

print("=" * 60)
print("MEJOR SOLUCIÓN ENCONTRADA")
print("=" * 60)
print(f"  Ruta (índices 0-{NUM_CITIES-1}): {best_route}")
print(f"  Ruta (ciudades 1-{NUM_CITIES}):  {[r+1 for r in best_route]}")
print(f"  Distancia total: {best_dist:.4f}")
print(f"  Aptitud (1/distancia): {best_fitness:.6f}")
print(f"\n  Distancia promedio inicial: {avg_distance_history[0]:.4f}")
print(f"  Distancia promedio final:   {avg_distance_history[-1]:.4f}")
print(f"  Mejora: {((avg_distance_history[0] - best_dist) / avg_distance_history[0] * 100):.2f}%")

---
## 9. Evolución de la distancia a lo largo de las generaciones

Se grafica:
- **a)** La distancia de la mejor ruta encontrada en cada generación
- **b)** La distancia promedio de la población en cada generación

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
generations = list(range(1, len(best_distance_history) + 1))
ax.plot(generations, best_distance_history, 'b-', linewidth=2, label='a) Mejor distancia')
ax.plot(generations, avg_distance_history, 'r-', linewidth=1.5, alpha=0.7, label='b) Distancia promedio')
ax.set_xlabel('Generación', fontsize=12)
ax.set_ylabel('Distancia total', fontsize=12)
ax.set_title(f'Evolución del AG para el TSP ({NUM_CITIES} ciudades)\n(Pob={POP_SIZE}, Pc={PC}, Pm={PM})', fontsize=14)
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 10. Visualización de la mejor ruta encontrada

In [ ]:
def plot_route(cities, route, title="Ruta del viajante", ax=None):

    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 8))

    # Dibujar la ruta
    route_list = list(route) + [route[0]]  # Cerrar el ciclo
    xs = [cities[int(i)][0] for i in route_list]
    ys = [cities[int(i)][1] for i in route_list]

    ax.plot(xs, ys, 'o-', color='steelblue', linewidth=2, markersize=8,
            markerfacecolor='red', markeredgecolor='black', markeredgewidth=1.5)

    for i, (x, y) in enumerate(cities):
        ax.annotate(f'{i+1}', (x, y), textcoords="offset points", xytext=(8, 8),
                    fontsize=10, fontweight='bold', color='darkblue')

    start_city = route[0]
    ax.plot(cities[start_city][0], cities[start_city][1], 'g^', markersize=15,
            zorder=5, label='Inicio')

    ax.set_xlabel('X', fontsize=12)
    ax.set_ylabel('Y', fontsize=12)
    dist = route_distance(route, dist_matrix)
    ax.set_title(f'{title}\nDistancia total: {dist:.2f}', fontsize=13)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='best', fontsize=10)

    return ax


print("Función de visualización definida.")

In [ ]:
plot_route(cities, best_route, title=f'Mejor ruta encontrada (Gen {NUM_GENERATIONS})')
plt.tight_layout()
plt.show()

---
## 11. Evolución visual de la mejor ruta en el tiempo

Se muestran las mejores rutas encontradas en diferentes generaciones,
permitiendo observar cómo el algoritmo mejora progresivamente la solución.

In [ ]:
logged_gens = sorted(best_routes_log.keys())

if len(logged_gens) > 6:
    indices = np.linspace(0, len(logged_gens) - 1, 6, dtype=int)
    selected_gens = [logged_gens[i] for i in indices]
else:
    selected_gens = logged_gens

if NUM_GENERATIONS not in selected_gens:
    selected_gens.append(NUM_GENERATIONS)
    best_routes_log[NUM_GENERATIONS] = best_route

selected_gens = sorted(set(selected_gens))

n_plots = len(selected_gens)
n_cols = 3
n_rows = math.ceil(n_plots / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 6 * n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1)

for idx, gen in enumerate(selected_gens):
    row = idx // n_cols
    col = idx % n_cols
    ax = axes[row, col]
    route = best_routes_log[gen]
    plot_route(cities, route, title=f'Generación {gen}', ax=ax)

for idx in range(n_plots, n_rows * n_cols):
    row = idx // n_cols
    col = idx % n_cols
    axes[row, col].set_visible(False)

plt.suptitle('Evolución de la mejor ruta a lo largo de las generaciones', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

---
## 12. Comparación de la ruta inicial vs. la ruta final

Se compara la primera ruta generada (aleatoria) con la mejor ruta encontrada
por el algoritmo genético.

In [ ]:
initial_route = initial_population[0]
initial_dist = route_distance(initial_route, dist_matrix)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

plot_route(cities, initial_route, title=f'Ruta inicial (aleatoria)\nDistancia: {initial_dist:.2f}', ax=axes[0])
plot_route(cities, best_route, title=f'Mejor ruta (Gen {NUM_GENERATIONS})\nDistancia: {best_dist:.2f}', ax=axes[1])

plt.suptitle('Comparación: Ruta inicial vs. Mejor ruta encontrada', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

improvement = ((initial_dist - best_dist) / initial_dist) * 100
print(f"\nMejora respecto a la ruta inicial: {improvement:.2f}%")
print(f"  Distancia inicial: {initial_dist:.2f}")
print(f"  Distancia final:   {best_dist:.2f}")
print(f"  Reducción:         {initial_dist - best_dist:.2f}")

---
## 13. Análisis de sensibilidad: Efecto de Pm y Pc

Se evalúa cómo cambian los resultados al variar los parámetros del AG.

In [ ]:
def run_tsp_experiment(pop_size=100, num_generations=300, pc=0.9, pm=0.1,
                      keep_elitism=2, seed=42):

    global best_distance_history, avg_distance_history, best_routes_log

    best_distance_history = []
    avg_distance_history = []
    best_routes_log = {}

    random.seed(seed)
    np.random.seed(seed)
    init_pop = []
    for _ in range(pop_size):
        individual = list(range(NUM_CITIES))
        random.shuffle(individual)
        init_pop.append(individual)
    init_pop = np.array(init_pop)

    ga = pygad.GA(
        num_generations=num_generations,
        num_parents_mating=pop_size // 2,
        initial_population=init_pop,
        fitness_func=fitness_tsp,
        parent_selection_type="rws",
        crossover_type=order_crossover,
        crossover_probability=pc,
        mutation_type=swap_mutation,
        mutation_probability=pm,
        keep_elitism=keep_elitism,
        allow_duplicate_genes=True,
        on_generation=on_generation_tsp,
        suppress_warnings=True,
        random_seed=seed,
        gene_type=int,
    )

    ga.run()

    best_sol, best_fit, _ = ga.best_solution()
    best_d = route_distance(best_sol, dist_matrix)

    return {
        'best_dist': best_d,
        'best_route': best_sol.astype(int),
        'best_dist_history': list(best_distance_history),
        'avg_dist_history': list(avg_distance_history),
    }

print("Función de experimento definida.")

### 13.1 Efecto de la Probabilidad de Cruce (Pc)

In [ ]:
pc_values_tsp = [0.9, 0.7, 0.5]
results_pc = {}

for pc in pc_values_tsp:
    print(f"Ejecutando con Pc={pc}...")
    result = run_tsp_experiment(pc=pc, pm=0.1, seed=42)
    results_pc[pc] = result
    print(f"  Mejor distancia: {result['best_dist']:.2f}")

print("\nExperimentos con diferentes Pc completados.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for pc in pc_values_tsp:
    r = results_pc[pc]
    gens = list(range(1, len(r['best_dist_history']) + 1))
    axes[0].plot(gens, r['best_dist_history'], linewidth=2, label=f'Pc={pc}')
axes[0].set_xlabel('Generación', fontsize=12)
axes[0].set_ylabel('Mejor distancia', fontsize=12)
axes[0].set_title('a) Mejor distancia (Pm=0.1)', fontsize=13)
axes[0].legend(loc='best', fontsize=10)
axes[0].grid(True, alpha=0.3)

for pc in pc_values_tsp:
    r = results_pc[pc]
    gens = list(range(1, len(r['avg_dist_history']) + 1))
    axes[1].plot(gens, r['avg_dist_history'], linewidth=2, label=f'Pc={pc}')
axes[1].set_xlabel('Generación', fontsize=12)
axes[1].set_ylabel('Distancia promedio', fontsize=12)
axes[1].set_title('b) Distancia promedio (Pm=0.1)', fontsize=13)
axes[1].legend(loc='best', fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Efecto de la Probabilidad de Cruce (Pc) en el TSP', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 13.2 Efecto de la Probabilidad de Mutación (Pm)

In [ ]:
pm_values_tsp = [0.05, 0.1, 0.3]
results_pm = {}

for pm in pm_values_tsp:
    print(f"Ejecutando con Pm={pm}...")
    result = run_tsp_experiment(pc=0.9, pm=pm, seed=42)
    results_pm[pm] = result
    print(f"  Mejor distancia: {result['best_dist']:.2f}")

print("\nExperimentos con diferentes Pm completados.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for pm in pm_values_tsp:
    r = results_pm[pm]
    gens = list(range(1, len(r['best_dist_history']) + 1))
    axes[0].plot(gens, r['best_dist_history'], linewidth=2, label=f'Pm={pm}')
axes[0].set_xlabel('Generación', fontsize=12)
axes[0].set_ylabel('Mejor distancia', fontsize=12)
axes[0].set_title('a) Mejor distancia (Pc=0.9)', fontsize=13)
axes[0].legend(loc='best', fontsize=10)
axes[0].grid(True, alpha=0.3)

for pm in pm_values_tsp:
    r = results_pm[pm]
    gens = list(range(1, len(r['avg_dist_history']) + 1))
    axes[1].plot(gens, r['avg_dist_history'], linewidth=2, label=f'Pm={pm}')
axes[1].set_xlabel('Generación', fontsize=12)
axes[1].set_ylabel('Distancia promedio', fontsize=12)
axes[1].set_title('b) Distancia promedio (Pc=0.9)', fontsize=13)
axes[1].legend(loc='best', fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.suptitle('Efecto de la Probabilidad de Mutación (Pm) en el TSP', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 14. Resumen y conclusiones

### Resultados principales

El algoritmo genético aplicado al Problema del Viajante de Comercio (TSP) con
**15 ciudades** logró encontrar una ruta considerablemente mejor que una selección
aleatoria, demostrando la capacidad de los algoritmos evolutivos para problemas
de optimización combinatoria.

### Observaciones clave

1. **Representación por permutación:** Fue esencial usar operadores especializados
   (Order Crossover, mutación por intercambio) para garantizar soluciones válidas.

2. **Convergencia:** El AG converge rápidamente en las primeras generaciones,
   realizando mejoras incrementales en las generaciones posteriores.

3. **Efecto de Pc:** Una probabilidad de cruce alta (0.9) favorece la exploración
   del espacio de búsqueda al combinar segmentos de rutas diferentes.

4. **Efecto de Pm:** Una mutación moderada (0.1) ayuda a escapar de óptimos locales,
   mientras que valores muy altos degradan el rendimiento al introducir demasiada
   aleatoriedad.

5. **Naturaleza NP-completa:** Para 15 ciudades, el espacio de búsqueda contiene
   $14!/2 \approx 4.36 \times 10^{10}$ posibles rutas. El AG explora eficientemente
   una fracción minúscula de este espacio y encuentra soluciones de buena calidad.

### Limitaciones

- El AG no garantiza encontrar la solución óptima global.
- Para un mayor número de ciudades, se requiere ajustar los parámetros y posiblemente
  incorporar técnicas de búsqueda local (meméticos) o enfriamiento simulado.